In [ ]:
!pip install -q unsloth
!pip install -q transformers accelerate bitsandbytes

In [2]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [3]:
from unsloth import FastLanguageModel

print("Unsloth loaded successfully")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth loaded successfully


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!unzip "/content/drive/MyDrive/ML_Music_Project/music_lora.zip"

Archive:  /content/drive/MyDrive/ML_Music_Project/music_lora.zip
   creating: music_lora/
  inflating: music_lora/chat_template.jinja  
  inflating: music_lora/tokenizer_config.json  
  inflating: music_lora/README.md    
  inflating: music_lora/tokenizer.json  
  inflating: music_lora/adapter_config.json  
  inflating: music_lora/adapter_model.safetensors  


In [6]:
import os

print(os.listdir("/content/music_lora"))

['chat_template.jinja', 'tokenizer_config.json', 'README.md', 'tokenizer.json', 'adapter_config.json', 'adapter_model.safetensors']


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=1024,
    load_in_4bit=True
)

In [8]:
model.load_adapter("/content/music_lora")

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

In [ ]:
FastLanguageModel.for_inference(model)

In [30]:
import sys
sys.path.append("/content/drive/MyDrive/ML_Music_Project")
!pip install -q scipy
from pipeline import load_audio_model, generate

In [14]:
processor, audio = load_audio_model("facebook/musicgen-small")
print("MusicGen loaded")

preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/7.87k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.37k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

MusicgenForConditionalGeneration LOAD REPORT from: facebook/musicgen-small
Key                                           | Status     |  | 
----------------------------------------------+------------+--+-
decoder.model.decoder.embed_positions.weights | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

MusicGen loaded


In [ ]:
from IPython.display import Audio, display
import html

DURATION = 15

prompts = [
    "Late night staring at the stars after a tough day",
    "sitting in a forest surrounded by pine trees and watching sunrise",
    "Shiv Tandav at late night",
    "happy music for celebrating a big win",
    "sad piano after a breakup",
    "cool synthwave for driving through the city at night",
    "marriage of my friend heavily drunk",
    "energetic afrobeats for a summer rooftop party",
    "epic orchestral music for a final boss battle",
    "in a vacation with family in a isolated village fishing",
]

for i, p in enumerate(prompts):
    print("=" * 80)
    print(f"[{i+1}/{len(prompts)}]  PROMPT: {p}")
    try:
        r = generate(p, model, tokenizer, processor, audio,
                     out_path=f"clip_{i}.wav", duration_s=DURATION)
        print("JSON        :", r["json"])
        print("AUDIO PROMPT:", r["audio_prompt"])
        display(Audio(r["wav"]))
    except Exception as e:
        print("FAILED:", e)
    print()